<a href="https://colab.research.google.com/github/javanmardi298/products_search/blob/main/02_llm_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Product search, part 2: LLM recommendations (Colab)

Uses the rankings from part 1 (`results/rankings.json`). The top 8 products go to a local LLM
(Ollama, running on the Colab GPU), which recommends up to 3.

- v1: dense search, the LLM answers in free text
- v2: hybrid search + reranker, the LLM answers in JSON and the product ids are checked

Runtime: T4 GPU. Models: `qwen2.5:3b` (recommends) and `qwen2.5:7b` (judge).

In [1]:
!pip install -q ollama gradio

In [2]:
!apt-get install -y -q zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import os

user = "javanmardi298"
repo = "products_search"

os.chdir("/content")
if not os.path.exists(repo):
    os.system(f"git clone https://github.com/{user}/{repo}.git")
else:
    os.system(f"cd {repo} && git pull")

if os.path.exists(repo + "/shop_common.py"):
    os.chdir(repo)
print(os.getcwd())
print(os.listdir())

/content/products_search
['shop_common.py', '.gitignore', '01_retrieval_colab.ipynb', '.git', 'split.json', 'results', 'requirements.txt', 'README.md', 'check_common.py', 'LICENSE']


In [4]:
import json
import re
import subprocess
import time

import numpy as np
import pandas as pd
from google.colab import files
from tqdm.auto import tqdm

from shop_common import load_data, features_to_text, ndcg, judged, bootstrap, short_name, SEED

LLM = "qwen2.5:3b"     # makes the recommendations
JUDGE = "qwen2.5:7b"   # checks them, should be bigger than LLM
N = 8                  # products the LLM sees
LIMIT = None           # e.g. 40 for a quick first run
RUN_TEST = False

os.makedirs("results", exist_ok=True)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

GPU 0: Tesla T4 (UUID: GPU-4a3cd7ae-1c8b-dca2-e5dc-606062887828)



In [5]:
server = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

for m in [LLM, JUDGE]:
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    print(m, "ok" if r.returncode == 0 else r.stderr)

qwen2.5:3b ok
qwen2.5:7b ok


## Data and results from Colab

In [6]:
for f in ["split.json", "results/rankings.json", "results/ablation_dev.csv"]:
    if not os.path.exists(f):
        print("missing:", f, "(get it from the Colab run)")

products, queries, labels, qrels, query_text = load_data()
prod = products.set_index("product_id")

with open("split.json") as f:
    split = json.load(f)
dev = split["dev"]
test = split["test"]
if LIMIT:
    dev = dev[:LIMIT]

with open("results/rankings.json") as f:
    raw = json.load(f)
rankings = {}
for name in raw:
    rankings[name] = {int(q): r for q, r in raw[name].items()}  # json keys are strings

print(len(dev), "dev queries")
pd.read_csv("results/ablation_dev.csv", index_col=0)

downloading product.csv
downloading query.csv
downloading label.csv
160 dev queries


,ndcg@10,recall@10,mrr,judged@10,ms/query
dense / basic,0.700,0.275,0.518,0.842,22.3
bm25 / basic,0.602,0.259,0.466,0.781,40.9
dense / rich,0.692,0.269,0.512,0.823,22.6
bm25 / rich,0.630,0.250,0.492,0.799,52.2
hybrid,0.706,0.268,0.543,0.853,108.0
hybrid + rerank,0.736,0.311,0.560,0.868,276.0


## Ollama

In [7]:
import ollama

client = ollama.Client(timeout=600)
options = {"temperature": 0, "seed": SEED, "num_ctx": 8192}  # the default context is too small


def ask(model, messages, format=None):
    for i in range(3):
        try:
            res = client.chat(model=model, messages=messages, options=options, format=format)
            return res.message.content
        except Exception as e:
            print("error, trying again:", e)
            time.sleep(5)
    raise Exception("Ollama is not answering")


print([m.model for m in client.list().models])

ask(LLM, [{"role": "user", "content": "hi"}])  # loads the model

start = time.time()
ask(LLM, [{"role": "user", "content": "Name three kinds of chairs."}])
print(f"{time.time() - start:.1f} s for a short answer")

['qwen2.5:7b', 'qwen2.5:3b']
2.3 s for a short answer


## Prompts

v1 sees name + description. v2 also sees category, rating and features.

In [8]:
def product_card(pid, rich):
    p = prod.loc[pid]
    if not rich:
        return f"[P{pid}] {p.product_name}: {p.product_description[:300]}"
    if pd.isna(p.average_rating):
        rating = "no ratings"
    else:
        rating = f"{p.average_rating:.1f}/5 ({p.rating_count:.0f} ratings)"
    return (f"[P{pid}] {p.product_name}\n"
            f"  category: {p.product_class} | {rating}\n"
            f"  features: {features_to_text(p.product_features, 8)}\n"
            f"  description: {p.product_description[:300]}")


print(product_card(rankings["v2"][dev[0]][0], rich=True))

[P33698] smart coffee table with storage
  category: Coffee & Cocktail Tables | 4.5/5 (903 ratings)
  features: style: modern & contemporary; overallheight-toptobottom: 17.7; dssecondaryproductstyle: ultra-modern; overallproductweight: 104; basetype: 4 legs; drawerinteriorwidth-sidetoside: 13; numberofdrawers: 3; topglasstype: tempered glass
  description: the sobro is a smart coffee table designed to support your connected lifestyle . with a refrigerated drawer , 2 bluetooth speakers , 2 usb charging ports , 4 110v outlets , and led lights , the sobro keeps you powered up . it could be the greatest living room innovation since the remote control ! th


In [9]:
v1_prompt = """You are a helpful shopping assistant for a home goods store.
Recommend up to 3 products for the customer and explain why.
Refer to products by ID, like P123."""


def ask_v1(query, cands):
    text = "\n\n".join(product_card(p, False) for p in cands)
    answer = ask(LLM, [
        {"role": "system", "content": v1_prompt},
        {"role": "user", "content": f"PRODUCTS:\n{text}\n\nCustomer: {query}"},
    ])
    picked = []
    for x in re.findall(r"\bP(\d+)\b", answer):
        if int(x) not in picked:
            picked.append(int(x))
    return {"text": answer, "picked": picked}

In [10]:
schema = {
    "type": "object",
    "properties": {
        "picks": {"type": "array", "maxItems": 3, "items": {
            "type": "object",
            "properties": {"id": {"type": "integer"}, "why": {"type": "string"}},
            "required": ["id", "why"]}},
        "nothing_fits": {"type": "boolean"},
    },
    "required": ["picks", "nothing_fits"],
}

v2_prompt = """You are a shopping assistant for a home goods store.
Pick up to 3 products from PRODUCTS that best match the customer's request.
- Only use products from the list. The id is the number after P.
- Every reason must be backed by that product's information.
- A product that matches every part of the request beats a partial match.
- If nothing fits, return no picks and set nothing_fits to true."""


def ask_v2(query, cands):
    text = "\n\n".join(product_card(p, True) for p in cands)
    answer = ask(LLM, [
        {"role": "system", "content": v2_prompt},
        {"role": "user", "content": f"PRODUCTS:\n{text}\n\nCustomer: {query}"},
    ], format=schema)
    try:
        out = json.loads(answer)
    except Exception:
        out = {"picks": [], "nothing_fits": False}

    picked, reasons = [], {}
    for pick in out.get("picks", []):
        try:
            pid = int(pick["id"])
        except Exception:
            continue
        picked.append(pid)
        reasons[pid] = pick.get("why", "")
    lines = [f"P{p}: {reasons[p]}" for p in picked]
    return {"text": "\n".join(lines), "picked": picked, "reasons": reasons,
            "nothing_fits": out.get("nothing_fits", False)}


systems = {"v1": ask_v1, "v2": ask_v2}

## Run v1 and v2



In [11]:
def run(name, qids):
    file = f"results/{name}_{short_name(LLM)}_answers.jsonl"
    done = set()
    if os.path.exists(file):
        with open(file) as f:
            for line in f:
                done.add(json.loads(line)["qid"])

    with open(file, "a") as f:
        for q in tqdm([q for q in qids if q not in done], desc=name):
            cands = rankings[name][q][:N]
            start = time.time()
            ans = systems[name](query_text[q], cands)
            ans.pop("reasons", None)
            ans["qid"] = q
            ans["cands"] = cands
            ans["valid"] = [p for p in ans["picked"] if p in cands][:3]
            ans["made_up"] = [p for p in ans["picked"] if p not in cands]
            ans["sec"] = round(time.time() - start, 2)
            f.write(json.dumps(ans) + "\n")

    return pd.read_json(file, lines=True).set_index("qid").loc[qids]


# first try 5 queries
answers = {}
for name in systems:
    answers[name] = run(name, dev[:5])
answers["v2"].head()

v1:   0%|          | 0/5 [00:00<?, ?it/s]

v2:   0%|          | 0/5 [00:00<?, ?it/s]

,text,picked,nothing_fits,cands,valid,made_up,sec
qid,,,,,,,
1,P33698: The customer requested a smart coffee ...,[33698],False,"[33698, 34862, 1308, 21580, 41696, 9929, 28935...",[33698],[],2.08
6,P33512: P33512 is a side chair in clear acryli...,"[33512, 41827, 29502]",False,"[33512, 29502, 41827, 32575, 41828, 29550, 219...","[33512, 41827, 29502]",[],2.58
9,"P7997: The product is a coffee table fire pit,...","[7997, 24419, 20647]",False,"[7997, 27916, 24419, 7509, 13615, 38098, 20647...","[7997, 24419, 20647]",[],2.38
12,P25477: P25477 matches the request for a large...,"[25477, 23580, 32789]",False,"[25477, 28501, 224, 11321, 23580, 32789, 41506...","[25477, 23580, 32789]",[],3.31
13,P24047: The customer requested an outdoor priv...,"[24047, 12509, 12508]",False,"[24047, 12509, 12508, 12510, 40883, 5944, 3787...","[24047, 12509, 12508]",[],3.74


In [12]:
for name in systems:
    answers[name] = run(name, dev)
print(len(answers["v1"]), len(answers["v2"]))

v1:   0%|          | 0/155 [00:00<?, ?it/s]

v2:   0%|          | 0/155 [00:00<?, ?it/s]

160 160


## Judge


In [13]:
judge_prompt = """You check a shopping assistant's answer for factual grounding.

PRODUCT INFORMATION (the only source of truth):
{facts}

ASSISTANT ANSWER:
{answer}

Is every factual claim about these products (material, size, color, features, rating, category)
supported by the product information? Ignore opinions like "great choice".
If not, quote the first unsupported claim."""

judge_schema = {
    "type": "object",
    "properties": {"verdict": {"type": "string", "enum": ["supported", "unsupported"]},
                   "claim": {"type": "string"}},
    "required": ["verdict", "claim"],
}


def check(answer, picked):
    facts = "\n\n".join(product_card(p, True) for p in picked)
    msg = [{"role": "user", "content": judge_prompt.format(facts=facts, answer=answer)}]
    return json.loads(ask(JUDGE, msg, format=judge_schema))


def judge_all(name, ans):
    file = f"results/{name}_{short_name(LLM)}_judged_by_{short_name(JUDGE)}.jsonl"
    done = set()
    if os.path.exists(file):
        with open(file) as f:
            for line in f:
                done.add(json.loads(line)["qid"])

    with open(file, "a") as f:
        for q, row in tqdm(list(ans.iterrows()), desc="judge " + name):
            if q in done:
                continue
            if len(row["valid"]) == 0:
                v = {"verdict": "no picks", "claim": ""}
            else:
                v = check(row["text"], row["valid"])
            v["qid"] = q
            f.write(json.dumps(v) + "\n")

    return pd.read_json(file, lines=True).set_index("qid").loc[ans.index]


verdicts = {}
for name in systems:
    verdicts[name] = judge_all(name, answers[name])

judge v1:   0%|          | 0/160 [00:00<?, ?it/s]

error, trying again: model runner has unexpectedly stopped, this may be due to resource limitations or an internal error, check ollama server logs for details (status code: 500)


judge v2:   0%|          | 0/160 [00:00<?, ?it/s]

## v1 vs v2

In [14]:
def per_query(name, qids):
    rows = []
    for q in qids:
        a = answers[name].loc[q]
        rel = qrels[q]
        ranked = rankings[name][q]
        valid = a["valid"]
        row = {
            "qid": q,
            "ndcg@10": ndcg(ranked, rel),
            "judged@10": judged(ranked, rel),
            "made-up ids": len(a["made_up"]) / max(1, len(a["picked"])),
            "no picks": len(valid) == 0,
            "sec": a["sec"],
        }
        if valid:
            row["picks exact"] = np.mean([rel.get(p) == 2 for p in valid])
            row["picks relevant"] = np.mean([rel.get(p, 0) >= 1 for p in valid])
        else:
            row["picks exact"] = np.nan
            row["picks relevant"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("qid").astype(float)


def summary(name, qids):
    s = per_query(name, qids)
    out = s.mean().round(3).to_dict()
    v = verdicts[name]
    v = v[v["verdict"] != "no picks"]
    out["faithful"] = round((v["verdict"] == "supported").mean(), 3)
    out["sec p95"] = round(s["sec"].quantile(0.95), 1)
    return out


compare = pd.DataFrame({"v1": summary("v1", dev), "v2": summary("v2", dev)})
compare.to_csv(f"results/systems_dev_{short_name(LLM)}.csv")
compare

,v1,v2
ndcg@10,0.700,0.736
judged@10,0.842,0.868
made-up ids,0.004,0.075
no picks,0.000,0.106
sec,4.313,2.882
picks exact,0.404,0.512
picks relevant,0.857,0.883
faithful,0.956,0.839
sec p95,5.300,4.000


In [15]:
s1 = per_query("v1", dev)
s2 = per_query("v2", dev)

sig = {}
for m in ["ndcg@10", "picks exact", "picks relevant"]:
    sig[m] = bootstrap(s1[m], s2[m])
pd.DataFrame(sig).T

,diff,low,high,p
ndcg@10,0.037,0.011,0.063,0.0020
picks exact,0.097,0.043,0.154,0.0001
picks relevant,0.015,-0.024,0.055,0.2332


## A closer look

Where in the 8 candidates do the picks come from? Does v2 say "nothing fits" at the right time? Which queries are hard?


In [16]:
position = {}
for name in systems:
    pos = []
    for q in dev:
        a = answers[name].loc[q]
        for p in a["valid"]:
            pos.append(a["cands"].index(p) + 1)
    position[name] = pd.Series(pos).value_counts().sort_index()

pd.DataFrame(position).fillna(0).astype(int)

,v1,v2
1,123,132
2,77,39
3,86,68
4,40,29
5,33,32
6,42,28
7,46,25
8,31,13


In [17]:
exact_in_cands = []
said_nothing = []
for q in dev:
    a = answers["v2"].loc[q]
    exact_in_cands.append(any(qrels[q].get(p) == 2 for p in a["cands"]))
    said_nothing.append(a["nothing_fits"] == True)

pd.crosstab(pd.Series(exact_in_cands, name="exact product in the 8"),
            pd.Series(said_nothing, name="v2 said nothing fits"))

v2 said nothing fits,False,True
exact product in the 8,,
False,52,5
True,101,2


In [18]:
qclass = dict(zip(queries["query_id"], queries["query_class"].fillna("")))
hard = s2.copy()
hard["words"] = [len(query_text[q].split()) for q in hard.index]
hard["class"] = [qclass.get(q, "") for q in hard.index]
hard["length"] = pd.cut(hard["words"], [0, 2, 4, 50], labels=["1-2 words", "3-4 words", "5+ words"])

print(hard.groupby("length", observed=True)["ndcg@10"].agg(["mean", "count"]).round(3))
by_class = hard.groupby("class")["ndcg@10"].agg(["mean", "count"])
by_class[by_class["count"] >= 3].sort_values("mean").head(10)

            mean  count
length                 
1-2 words  0.820     39
3-4 words  0.721     95
5+ words   0.669     26


,mean,count
class,,
Outdoor Conversation Sets,0.301778,3
,0.394470,3
Cabinet and Drawer Pulls,0.481596,3
Accent Chairs,0.552577,3
Wall Art,0.585758,7
Beds,0.716761,5
Sofas,0.734255,3
Dining Chairs,0.754445,3
Kids Beds,0.771501,3


## Can I trust the judge?

I label 30 of the judge's answers myself (half supported, half unsupported).
The first cell makes the csv, I fill `my_label` on my laptop, the second cell uploads it and computes kappa.

In [19]:
from sklearn.metrics import cohen_kappa_score

file = "results/judge_review.csv"
if not os.path.exists(file):
    df = answers["v2"].join(verdicts["v2"])
    df = df[df["verdict"] != "no picks"]
    parts = []
    for v, group in df.groupby("verdict"):
        parts.append(group.sample(min(15, len(group)), random_state=SEED))
    sample = pd.concat(parts)
    sample["query"] = [query_text[q] for q in sample.index]
    sample["products"] = [" || ".join(product_card(p, True) for p in ps) for ps in sample["valid"]]
    sample["my_label"] = ""
    sample[["query", "text", "products", "verdict", "claim", "my_label"]].to_csv(file)
    files.download(file)
    print("fill my_label (supported / unsupported) on your laptop, then run the next cell")
else:
    print(file, "already exists")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

fill my_label (supported / unsupported) on your laptop, then run the next cell


In [20]:
uploaded = files.upload()  # choose the filled judge_review.csv
for name in uploaded:
    os.replace(name, file)

review = pd.read_csv(file)
review = review[review["my_label"].notna()]
if len(review) == 0:
    print("my_label is empty, fill it first")
else:
    review["my_label"] = review["my_label"].str.strip().str.lower()
    agree = (review["my_label"] == review["verdict"]).mean()
    kappa = cohen_kappa_score(review["my_label"], review["verdict"])
    print(f"{len(review)} checked, agreement {agree:.2f}, kappa {kappa:.2f}")
    print(pd.crosstab(review["my_label"], review["verdict"]))

Saving judge_review.csv to judge_review.csv
my_label is empty, fill it first


## Error analysis

The 20 worst v2 queries. I fill the `error type` column by hand: vocabulary gap, missed attribute,
wrong category, good but no label, LLM picked badly, unclear query.

In [21]:
label_name = {2: "Exact", 1: "Partial", 0: "Irrelevant"}

rows = []
for q in dev:
    rel = qrels[q]
    ranked = rankings["v2"][q]
    top3 = [f"{prod.loc[p, 'product_name']} [{label_name.get(rel.get(p), 'no label')}]" for p in ranked[:3]]
    exact = [p for p in rel if rel[p] == 2]
    rows.append({
        "qid": q,
        "query": query_text[q],
        "ndcg@10": round(ndcg(ranked, rel), 3),
        "top 3": " | ".join(top3),
        "an exact product": prod.loc[exact[0], "product_name"] if exact else "none",
        "llm answer": answers["v2"].loc[q, "text"],
        "error type": "",
    })
worst = pd.DataFrame(rows).sort_values("ndcg@10").head(20)

file = "results/error_analysis.csv"
if not os.path.exists(file):
    worst.to_csv(file, index=False)
    files.download(file)
    print("fill the 'error type' column on your laptop, then run the next cell")
worst.head(10)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

fill the 'error type' column on your laptop, then run the next cell


,qid,query,ndcg@10,top 3,an exact product,llm answer,error type
27,83,wall art fiji,0.000,fiji 12 ft. w x 12 ft. d vinyl pergola [no lab...,none,P26404: The customer requested wall art relate...,
58,174,star wars rug,0.000,disney star wars disney star wars darth vader ...,none,P23953: The customer requested a 'star wars ru...,
61,179,promo codes or discounts,0.000,smartcode 955 commercial grade electronic door...,none,,
107,304,merlyn 6,0.000,merlyn 6-piece patio furniture set corner sofa...,none,P30336: P30336 matches the customer's request ...,
93,277,barstool patio sets,0.022,heathcote 5-piece bar set [no label] | hallsvi...,"2 piece outdoor swivel high bar stools , patio...",P35527: Braunstone 7 piece bar set offers dura...,
53,161,accent chair recliner,0.028,melo 29.52 '' wide tufted lounge chair and ott...,26 '' wide chenille armchair,P29759: P29759 melo 29.52 '' wide tufted loung...,
148,448,bed side table,0.037,plain end table [no label] | butner end table ...,1 - drawer nightstand,P4: P23842 hajrah end table with storage is sp...,
77,222,small loving roomtables,0.085,creeksville solid wood 2 piece nesting tables ...,abernethy solid wood end table,P1912: P1912 matches the customer's request fo...,
28,85,3/4 size mattress,0.110,mattress pad [no label] | gossage 3m polyester...,none,P1982: P1982 is a mattress pad that is suitabl...,
151,460,small wardrobe grey,0.169,pella armoire [Partial] | isenhour 55.1 '' w c...,harmony wooden armoire,P22065: P22065 is a small wardrobe (armoire) t...,


In [22]:
uploaded = files.upload()  # choose the filled error_analysis.csv
for name in uploaded:
    os.replace(name, file)

print(pd.read_csv(file)["error type"].dropna().value_counts())

Saving error_analysis.csv to error_analysis.csv
Series([], Name: count, dtype: int64)


## Test set

Only at the end, after all decisions were made on dev. Set `RUN_TEST = True` in the settings cell.

In [23]:
if RUN_TEST:
    for name in systems:
        answers[name] = run(name, test)
        verdicts[name] = judge_all(name, answers[name])
    final = pd.DataFrame({"v1": summary("v1", test), "v2": summary("v2", test)})
    final.to_csv(f"results/systems_test_{short_name(LLM)}.csv")
    t1, t2 = per_query("v1", test), per_query("v2", test)
    print(pd.DataFrame({m: bootstrap(t1[m], t2[m]) for m in ["ndcg@10", "picks exact"]}).T)
    print(final)
else:
    print("RUN_TEST is False, still working on dev")

RUN_TEST is False, still working on dev


## Versions

In [24]:
import platform
from importlib.metadata import version

models = {}
for m in client.list().models:
    if m.model in [LLM, JUDGE]:
        models[m.model] = m.digest[:12]

info = {"python": platform.python_version(), "ollama": version("ollama"), "pandas": version("pandas"),
        "models": models, "products shown": N, "dev queries": len(dev)}
with open("results/env_llm.json", "w") as f:
    json.dump(info, f, indent=2)
info

{'python': '3.13.16',
 'ollama': '0.6.3',
 'pandas': '2.2.3',
 'models': {'qwen2.5:7b': '845dbda0ea48', 'qwen2.5:3b': '357c53fb659c'},
 'products shown': 8,
 'dev queries': 160}

## Demo

v1 and v2 side by side on the WANDS queries.

In [26]:
import gradio as gr

all_queries = sorted(set(query_text[q] for q in dev + test))
qid_of = {query_text[q]: q for q in dev + test}


def demo(query):
    q = qid_of[query]
    out = []
    for name in systems:
        cands = rankings[name][q][:N]
        ans = systems[name](query, cands)
        if name == "v1":
            out.append(ans["text"])
            continue
        lines = []
        for p, why in ans["reasons"].items():
            if p in cands:
                lines.append(f"- **{prod.loc[p, 'product_name']}** ({prod.loc[p, 'product_class']}): {why}")
        if ans["nothing_fits"] or not lines:
            out.append("Nothing in the catalog fits this request well.")
        else:
            out.append("\n".join(lines))
    return out


gr.Interface(demo, gr.Dropdown(all_queries, label="Query"),
             [gr.Markdown(label="v1"), gr.Markdown(label="v2")], title="v1 vs v2").launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eb22d1995fcb8d3d30.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:
os.system("zip -r /content/colab_llm_results.zip results")
files.download("/content/colab_llm_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>